In [19]:

import os
import pandas as pd
from ucimlrepo import fetch_ucirepo 

def heart_failure(data_set_id):
    heart_failure_clinical_records = fetch_ucirepo(id=519) 
    X = heart_failure_clinical_records.data.features 
    y = heart_failure_clinical_records.data.targets
    df = pd.concat([X,y], axis=1)
    return(df, X, y)

df, X, y = heart_failure(519)

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
import joblib

numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'death_event' in numerical_features:
    numerical_features.remove('death_event')

ct_preprocessing = ColumnTransformer(
    [('numerical', StandardScaler(), numerical_features)]
)

pipeline = Pipeline([
    ('preprocessor', ct_preprocessing),
    ('model', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

X_test, X_train, y_test, y_train = train_test_split(X, y, test_size=0.2, random_state=42)

y_train=y_train.values.ravel()
y_test=y_test.values.ravel()


tuned_parameters = {
    'model__max_depth': [50],
    'model__n_estimators': [100],
    'model__min_samples_split': [4],
    'model__min_samples_leaf': [2],
    'model__max_features': ['sqrt']
}

tuned_model_rfc = GridSearchCV(pipeline,tuned_parameters, cv=5, error_score='raise' )
tuned_model_rfc.fit(X_train, y_train)

y_pred_1 = tuned_model_rfc.predict(X_test)
y_pred_1 = y_pred_1.ravel()

print("Best params:",tuned_model_rfc.best_params_)
print("Best score:",tuned_model_rfc.best_score_)

joblib.dump(tuned_model_rfc, 'rfc_model.pkl')

Best params: {'model__max_depth': 50, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 4, 'model__n_estimators': 100}
Best score: 0.7


['rfc_model.pkl']

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd
from ucimlrepo import fetch_ucirepo 
import joblib

def heart_failure(data_set_id):
    heart_failure_clinical_records = fetch_ucirepo(id=519) 
    X = heart_failure_clinical_records.data.features 
    y = heart_failure_clinical_records.data.targets
    df = pd.concat([X,y], axis=1)
    return(df, X, y)

df, X, y = heart_failure(519)

numerical_features = df.select_dtypes(['float64', 'int64']).columns.tolist()
if 'death_event' in numerical_features:
    numerical_features.remove('death_event')

ct_preprocessing = ColumnTransformer(
    [('numerical', StandardScaler(), numerical_features)]
)
pipeline_2 = Pipeline([
    ('preprocessor', ct_preprocessing),
    ('model', XGBClassifier(random_state=42))
])

X_test, X_train, y_test, y_train = train_test_split(X,y, test_size=0.2, random_state=42)
y_train=y_train.values.ravel()
y_test=y_test.values.ravel()

tuned_parameters_2 = {
    'model__max_depth':[25, 50, 75],
    'model__learning_rate':[0.5, 0.8, 0.9, 1],
    'model__n_estimators':[150, 165, 180, 190],
    'model__subsample':[0.7, 0.9, 1.0],
    'model__colsample_bytree': [0.4, 0.5, 0.6, 0.7, 0.8]
}

tuned_model_xgb = RandomizedSearchCV(pipeline_2,tuned_parameters_2, cv=5, n_iter=50, error_score='raise' )
tuned_model_xgb.fit(X_train, y_train)

y_pred_2 = tuned_model_xgb.predict(X_test)
y_pred_2 = y_pred_2.ravel()

print("Best params:",tuned_model_xgb.best_params_)
print("Best score:",tuned_model_xgb.best_score_)

joblib.dump(tuned_model_xgb, 'xgb_model.pkl')

Best params: {'model__subsample': 0.9, 'model__n_estimators': 165, 'model__max_depth': 50, 'model__learning_rate': 0.9, 'model__colsample_bytree': 0.8}
Best score: 0.7833333333333333


['xgb_model.pkl']

In [22]:
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier


rfc = RandomForestClassifier(random_state=42)
xgb = XGBClassifier(random_state=42)

voting = VotingClassifier(
        estimators=[('rfc', rfc), ('xgb', xgb)],
        voting='soft'
)

voting.fit(X_train, y_train)

predict_voting_y = voting.predict(X_test)

In [23]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_1))
print("Classification Report:")
print(classification_report(y_test, y_pred_1))

print("Confusion Matrix (Voting):")
print(confusion_matrix(y_test, predict_voting_y))
print("Classification Report (Voting):")
print(classification_report(y_test, predict_voting_y))

Confusion Matrix:
[[141  27]
 [ 14  57]]
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.84      0.87       168
           1       0.68      0.80      0.74        71

    accuracy                           0.83       239
   macro avg       0.79      0.82      0.80       239
weighted avg       0.84      0.83      0.83       239

Confusion Matrix (Voting):
[[139  29]
 [ 20  51]]
Classification Report (Voting):
              precision    recall  f1-score   support

           0       0.87      0.83      0.85       168
           1       0.64      0.72      0.68        71

    accuracy                           0.79       239
   macro avg       0.76      0.77      0.76       239
weighted avg       0.80      0.79      0.80       239



In [25]:
from pydantic import BaseModel
import joblib

from fastapi import FastAPI, Request

app = FastAPI(title="Heart Failure Prediction API", description="API for predicting heart failure using machine learning models.", version="1.0")

model=joblib.load('rfc_model.pkl')

class PatientData(BaseModel):
    age: float
    anaemia: int
    creatinine_phosphokinase: float
    diabetes: int
    ejection_fraction: float
    high_blood_pressure: int
    platelets: float
    serum_creatinine: float
    serum_sodium: float
    sex: int
    smoking: int
    time: float

@app.post("/predict")
async def predict_heart_failure(request: PatientData):
    input_data = pd.DataFrame([request.dict()])
    prediction = model.predict(input_data)
    probability = model.predict_proba(input_data)
    return {"prediction for death": int(prediction[0]), "probability of survival": probability[0].tolist()}

@app.get("/")
async def root():
    return {"message": "Welcome to the Heart Failure Prediction API. Use the /predict endpoint to make predictions."}